In [4]:
!pip install langchain-community
import os
import numpy as np
from langchain_community.embeddings import HuggingFaceBgeEmbeddings, OpenAIEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFaceEndpoint
from langchain_huggingface.llms import HuggingFacePipeline



In [3]:
!pip install langchain-huggingface

In [6]:
from google.colab import files

uploaded = files.upload()

Saving resume.zip to resume.zip


In [7]:
import zipfile

with zipfile.ZipFile("resume.zip", 'r') as zip_ref:
    zip_ref.extractall("resume")  # extracts files inside "resume" folder

In [11]:
import os

print("Root files/folders:", os.listdir())
print("Files in shown_resumes:", os.listdir("resume"))

Root files/folders: ['.config', 'resume', 'resume.zip', 'requirements.txt', 'sample_data']
Files in shown_resumes: ['resumes']


In [12]:
from urllib.request import urlretrieve

In [49]:
files = [
   "file:///C:/Users/admin/Downloads/Resumes_Part1%20(1).pdf",
   "file:///C:/Users/admin/Downloads/Resumes_Part2.pdf",
   "file:///C:/Users/admin/Downloads/Resumes_Part3.pdf",
   "file:///C:/Users/admin/Downloads/Resumes_Part4.pdf",
   "file:///C:/Users/admin/Downloads/Resumes_Part6%20(1).pdf",
   "file:///C:/Users/admin/Downloads/Resumes_Part7.pdf",
   "file:///C:/Users/admin/Downloads/Resume_8.pdf",
   "file:///C:/Users/admin/Downloads/Resume_9.pdf",
   "file:///C:/Users/admin/Downloads/Resume_10.pdf",
   "file:///C:/Users/admin/Downloads/Resume_10.pdf",
   "file:///C:/Users/admin/Downloads/Resume_13.pdf",
   "file:///C:/Users/admin/Downloads/Resume_14%20(1).pdf",
   "file:///C:/Users/admin/Downloads/Resume_15.pdf",
]
os.makedirs('resume',exist_ok=True)

In [51]:
import os
import fitz

pdf_dir = "resume"
pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith(".pdf")]

for file in pdf_files:
    file_path = os.path.join(pdf_dir, file)
    doc = fitz.open(file_path)  # <-- Use this relative path only
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()

    print(f"\nExtracted text from {file}:\n{text[:300]}...\n")

In [15]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 102.8 MB/s eta 0:00:00


In [52]:
loader=PyPDFDirectoryLoader('resume')

In [53]:
docs_before_split=loader.load()

In [20]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.7/309.7 kB 20.9 MB/s eta 0:00:00


In [54]:
len(docs_before_split[0].page_content)

182

In [55]:
import os

pdf_dir = "resumes"
pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith(".pdf")]
print("PDF files found:", pdf_files)

PDF files found: []


In [40]:
from langchain.document_loaders import PyPDFLoader
import os

folder = "resume"
docs_before_split = []

for filename in os.listdir(folder):
    if filename.lower().endswith(".pdf"):
        path = os.path.join(folder, filename)  # use filename with %20 as is
        print(f"Loading: {path}")
        loader = PyPDFLoader(path)
        docs = loader.load()
        docs_before_split.extend(docs)
print(f"Total pages loaded: {docs_before_split}")

Total pages loaded: []


In [56]:
text_splitter =  RecursiveCharacterTextSplitter(
    chunk_size =170,
    chunk_overlap = 20
)
docs_after_split = text_splitter.split_documents(docs_before_split)

In [57]:
len(docs_after_split[0].page_content)

129

In [58]:
avg_doc_length = lambda docs: sum([len(doc.page_content) for doc in docs])//len(docs)

In [59]:
avg_char_before_split = avg_doc_length(docs_before_split)
avg_char_after_split = avg_doc_length(docs_after_split)

In [60]:
print(f'before split: {avg_char_before_split}')
print(f'after split: {avg_char_after_split}')

before split: 184
after split: 99


In [61]:
huggingface_embedings=HuggingFaceBgeEmbeddings(
    model_name= "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs= {'device':'cpu'},
    encode_kwargs= {"normalize_embedings": True}
)

/tmp/ipython-input-61-2567111096.py:1: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  huggingface_embedings=HuggingFaceBgeEmbeddings(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 29.3 MB/s eta 0:00:00


In [64]:
vector_store=FAISS.from_documents(docs_after_split,huggingface_embedings)

In [65]:
query="Find candidates with TensorFlow + AWS experience?"

In [66]:
relevant_document=vector_store.similarity_search(query)

In [67]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={'k':3})

In [ ]:
access_token="****"


In [79]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

def setup_llm():
    model_id = "google/flan-t5-small"

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

    pipe = pipeline(
        "text2text-generation",
        model=model,
        tokenizer=tokenizer,
        max_length=128,
        do_sample=False,
    )
    return pipe

In [75]:
from langchain.prompts import PromptTemplate
prompt_template = """Use the following pieces of context to answer the question at the end. Please follow the following rules:
1. If you don't know the answer, don't try to make up an answer. Just say "I can't find the final answer but you may want to check the following links".
2. If you find the answer, write the answer in a concise way with five sentences maximum.

{context}

Question: {question}

Helpful Answer:
"""

PROMPT = PromptTemplate(
 template=prompt_template, input_variables=["context", "question"]
)

In [77]:
pip install -U langchain-huggingface

In [82]:
from langchain_huggingface import HuggingFacePipeline


In [80]:
from langchain.llms import HuggingFacePipeline
from transformers import pipeline

# Create the pipeline
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=512)

# Wrap it
llm = HuggingFacePipeline(pipeline=pipe)

Device set to use cuda:0


In [83]:
pipe = setup_llm()  # your HuggingFace pipeline (Text2TextGenerationPipeline)
llm = HuggingFacePipeline(pipeline=pipe)  # wrap it as a LangChain Runnable

Device set to use cuda:0


In [84]:
from langchain.chains import RetrievalQA

retrievalQA = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

USE the RAG

In [85]:
result = retrievalQA.invoke({"query" : query})
print(result)

{'query': 'Find candidates with TensorFlow + AWS experience?', 'result': 'ML Engineer', 'source_documents': [Document(id='a1f8335b-9343-482b-ba75-45426c5da54f', metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20250723120639', 'source': 'resume/resumes/Resumes_Part2.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Bob Lee\nCloud ML Engineer\nSkills: AWS, TensorFlow, Keras, Python\nExperience:\n- Designed AI solutions using TensorFlow models on AWS'), Document(id='58aaae29-9a43-4ae7-a22e-4cd3e3f9276a', metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20250723121831', 'source': 'resume/resumes/Resumes_Part3.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='Emily Davis\nSenior Data Scientist\nSkills: TensorFlow, AWS Sagemaker, SQL\nExperience:\n- Led team developing ML models in TensorFlow'), Document(id='87fe1233-003b-4c49-9475-154a28f